### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.generics:GenericModel` has been moved to `pydantic.BaseModel`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    logged_data=df,
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 292.27it/s]


2025-06-10 15:35:18.555 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:736 - Data batch-empirical estimation of propensity score.


2025-06-10 15:35:18.563 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:786 - Data prediction of expected reward based on gbm model.


In [6]:
evaluator.evaluate(mab=mab, visualize=True, n_mc_experiments=1000)

2025-06-10 15:35:18.910 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:882 - Data prediction of expected policy based on Monte Carlo experiments.


0it [00:00, ?it/s]

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(


...

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(
Sampling: [bias_0, out, weight_0]


2025-06-10 15:37:01.432 | INFO     | pybandits.offline_policy_evaluator:evaluate:955 - Offline Policy Evaluation for reward_0.


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/home/runner/work/pybandits/pybandits/pybandits/offline_policy_estimator.py:116: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  bootstrap_result = bootstrap(


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.493927,0.459391,0.527079,0.017125,b-ipw,reward_0
1,0.505308,0.499414,0.511350,0.003058,dm,reward_0
2,0.486250,0.454233,0.517930,0.016111,dr,reward_0
3,0.505308,0.499318,0.511449,0.003080,dros-opt,reward_0
4,0.486250,0.453397,0.518173,0.016371,dros-pess,reward_0
5,0.485801,0.453348,0.517507,0.016354,ipw,reward_0
6,0.000000,NaN,NaN,0.000000,rep,reward_0
7,0.486255,0.454412,0.517962,0.016303,sndr,reward_0
8,0.485671,0.454768,0.518444,0.016377,snips,reward_0
9,0.486250,0.455534,0.518586,0.016165,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, visualize=True, n_mc_experiments=1000)

2025-06-10 15:37:02.633 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1034 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(
Initializing NUTS using adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [weight_0, bias_0]


Sampling 2 chains for 500 tune and 1_000 draw iterations (1_000 + 2_000 draws total) took 7 seconds.


...

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pymc/data.py:384: FutureWarning: Data is now always mutable. Specifying the `mutable` kwarg will raise an error in a future release
  warnings.warn(


Sampling: [bias_0, out, weight_0]


2025-06-10 15:39:05.411 | INFO     | pybandits.offline_policy_evaluator:evaluate:955 - Offline Policy Evaluation for reward_0.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.497180,0.446101,0.552891,0.027196,b-ipw,reward_0
1,0.508102,0.502182,0.514272,0.003101,dm,reward_0
2,0.468805,0.425611,0.512288,0.021996,dr,reward_0
3,0.508102,0.502263,0.514187,0.003080,dros-opt,reward_0
4,0.468805,0.426562,0.512530,0.022039,dros-pess,reward_0
5,0.466491,0.416750,0.519966,0.026170,ipw,reward_0
6,0.444444,0.111111,1.111111,0.221809,rep,reward_0
7,0.468811,0.425714,0.511769,0.021885,sndr,reward_0
8,0.466414,0.416658,0.518317,0.026004,snips,reward_0
9,0.468805,0.424942,0.510938,0.022022,sg-dr,reward_0
